# Toy Experiment B — recovery across C-modes

**Question.** Which parameterizations of θ = (C, Λ) are recoverable at all, and
how does precision degrade with parameter count?

| mode | C | Λ params | why it matters |
|---|---|---|---|
| `k1` | continuous scalar, fixed | 1 | baseline (already passed) |
| `k2_strong` | continuous 2-col, fixed | 3 | **the key case** — unblocks the multi-mode model |
| `k2_weak` | continuous 2-col, fixed | 3 | when does a weak 2nd mode become undetectable? |
| `k3` | continuous 3-col, fixed | 6 | precision vs parameter count |
| `types5` | one-hot, 5 types, fixed | 15 | discrete case |
| `freeC` | learned | 1 + N | ceiling: best rank-1 with no prior |

**Why k=2 is the one to run first.** The 2D continuous coordinate is currently
blocked in the *inherited-α* regime because Hi-C PC2 does not align with
MiChroM's second demixing axis (r = −0.298). Under the optimizer that
misalignment is irrelevant — Λ is fitted from scratch, so PC2 need not align with
anything. This experiment answers with known ground truth whether a second mode
is recoverable at all, or intrinsically flat.

**Note on gauge.** With C *fixed* and of full column rank, Λ is uniquely
determined by M — there is no gauge freedom. Gauge only bites for `freeC`. So a
k=2 fixed-C failure would mean the *data* cannot determine the second mode, not
that the parameterization is ambiguous. That is exactly the question being asked.

---
### Cost

Measured rate ≈ 1.2M steps/min at N=300. Per mode: target (20×3M) ≈ 50 min,
ceiling (2×10×3M) ≈ 50 min, fit ≈ 20–40 min → **~2.3 h/mode**. Run overnight, or
subset `MODES_TO_RUN`. Everything is cached and resumable.

## 0. Setup

In [ ]:
from google.colab import drive
!git clone -q https://github.com/darinddv/chromatin_potential.git
!pip install -q "openmm[cuda12]" OpenMiChroM
import sys; sys.path.insert(0, '/content/chromatin_potential/src')
drive.mount('/content/drive')

import os, glob, json, time, inspect
import numpy as np
import matplotlib.pyplot as plt

from chromatin_potential.model import Model
from chromatin_potential.simulator import Simulator, BackgroundStack, SaveSpec
from chromatin_potential import optimizer as O

_src = inspect.getsource(O)
_msrc = inspect.getsource(Model)
checks = {
    'gradient sign fixed' : 'P_exp - P_sim' in _src,
    'best-iterate return' : 'best_params' in _src,
    'overshoot detection' : 'worsen_window' in _src,
    'lambda-collapse guard': 'collapse_tol' in _src,
    'gauge_fix implemented': 'unit_variance' in _msrc,
}
for k,v in checks.items(): print(f'  {"OK " if v else "MISSING"}  {k}')
assert all(checks.values()), 'modules out of date — pull and RESTART the runtime'
print('setup OK')

In [ ]:
RESET_LEVEL = 0          # 0 reuse | 1 redo fits | 2 redo everything

DATA = '/content/drive/MyDrive/uky_cheng/tecsas_ablation'
OUT  = f'{DATA}/toyB_cmodes'
os.makedirs(f'{OUT}/fits', exist_ok=True)

N_BEADS  = 300
names    = [f't{i:05d}' for i in range(N_BEADS)]
SEQ      = f'{OUT}/perbead_{N_BEADS}.txt'
with open(SEQ,'w') as fh:
    for i,n in enumerate(names): fh.write(f'{i+1} {n}\n')

# standing protocol (toy plan §1)
TARGET_REPLICAS = 20
TARGET_STEPS    = 3_000_000
CEIL_REPLICAS   = 10

def purge(pats, label):
    n=0
    for p in pats:
        for f in glob.glob(p): os.remove(f); n+=1
    print(f'  purged {n} files ({label})')

if RESET_LEVEL >= 2:
    purge([f'{OUT}/maps/*', f'{OUT}/traj/*', f'{OUT}/meta/*',
           f'{OUT}/fits/*', f'{OUT}/*.npy', f'{OUT}/*.json'], 'everything')
elif RESET_LEVEL == 1:
    purge([f'{OUT}/maps/*_it*', f'{OUT}/traj/*_it*', f'{OUT}/meta/*_it*',
           f'{OUT}/fits/*'], 'fits only — targets and ceilings kept')
else:
    print('  reusing cached artifacts')

sim = Simulator(seq_file=SEQ, out_dir=OUT, platform='cuda',
                background=BackgroundStack(), quiet=True)
print('simulator ready (quiet=True: OpenMiChroM banner suppressed)')

## 1. Define the modes

Each mode supplies a **true model** and a deliberately-wrong **initial model**.
Λ values are chosen so the leading mode matches the k=1 case (λ₁ = −1.2), making
precision comparable across modes.

`k2_weak` has a second mode 12× weaker than the first — the point is to find
where a mode stops being determinable.

In [ ]:
def coord(k, seed): return O.synthetic_coordinate(N_BEADS, k=k, n_domains=12, seed=seed)

def onehot_types(n_types=5, seed=5):
    seq  = O.synthetic_types(N_BEADS, types=tuple(f'T{i}' for i in range(n_types)),
                             mean_run=12, seed=seed)
    uniq = sorted(set(seq)); idx = {t:i for i,t in enumerate(uniq)}
    C = np.zeros((N_BEADS, len(uniq)))
    for i,t in enumerate(seq): C[i, idx[t]] = 1.0
    return C, seq, uniq

C_types, type_seq, type_names = onehot_types()
rng = np.random.default_rng(7)
LAM_TYPES = rng.normal(-0.30, 0.06, (len(type_names), len(type_names)))
LAM_TYPES = 0.5*(LAM_TYPES + LAM_TYPES.T)

MODES = {
 'k1'        : dict(C=coord(1,1), Lam=np.array([[-1.2]]),
                    init_Lam=np.array([[-0.3]]), free=False,
                    note='baseline scalar coordinate'),
 'k2_strong' : dict(C=coord(2,1), Lam=np.array([[-1.2, 0.30],[0.30, -0.60]]),
                    init_Lam=np.eye(2)*-0.3, free=False,
                    note='2nd mode half the 1st — should be recoverable'),
 'k2_weak'   : dict(C=coord(2,1), Lam=np.array([[-1.2, 0.05],[0.05, -0.10]]),
                    init_Lam=np.eye(2)*-0.3, free=False,
                    note='2nd mode 12x weaker — detectability test'),
 'k3'        : dict(C=coord(3,1),
                    Lam=np.array([[-1.2,0.20,0.10],[0.20,-0.60,0.05],[0.10,0.05,-0.30]]),
                    init_Lam=np.eye(3)*-0.3, free=False,
                    note='precision vs parameter count'),
 'types5'    : dict(C=C_types, Lam=LAM_TYPES,
                    init_Lam=np.full_like(LAM_TYPES, -0.20), free=False,
                    note='discrete one-hot, 5 types'),
 'freeC'     : dict(C=coord(1,1), Lam=np.array([[-1.2]]),
                    init_Lam=None, free=True,
                    note='CEILING: best rank-1 with no prior (not transferable)'),
}

MODES_TO_RUN = ['k2_strong']     # start here; add more as time allows

for name, m in MODES.items():
    k = m['C'].shape[1]; npar = k*(k+1)//2
    flag = '  <-- queued' if name in MODES_TO_RUN else ''
    print(f'  {name:11s} k={k}  Lambda params={npar:2d}  {m["note"]}{flag}')

## 2. Run

Per mode: generate target → measure noise ceiling → fit → report. Each stage is
cached separately, so an interrupted run resumes where it stopped.

In [ ]:
RESULTS_F = f'{OUT}/results.json'
results = json.load(open(RESULTS_F)) if os.path.exists(RESULTS_F) else {}

def run_mode(name):
    spec = MODES[name]
    C_true, Lam_true = spec['C'], spec['Lam']
    true_model = Model(C_true, Lam_true, c=-0.30, names=names)
    k = C_true.shape[1]
    t0 = time.time()
    print(f'\n=== {name}  (k={k}, {spec["note"]}) ===')

    # --- target ---
    tgt_f = f'{OUT}/target_{name}.npy'
    if os.path.exists(tgt_f):
        target = np.load(tgt_f); print('  target: cached')
    else:
        print('  target: generating (20 x 3M) ...')
        target = O.make_synthetic_target(sim, true_model,
                                         n_replicas=TARGET_REPLICAS,
                                         n_production=TARGET_STEPS,
                                         cond=f'truth_{name}', verbose=False)
        np.save(tgt_f, target)

    # --- noise ceiling ---
    ceil_f = f'{OUT}/ceiling_{name}.json'
    if os.path.exists(ceil_f):
        ceiling = json.load(open(ceil_f))['ceiling']; print(f'  ceiling: cached {ceiling:.5f}')
    else:
        print('  ceiling: measuring ...')
        ceiling,_,_ = O.replica_noise_ceiling(sim, true_model,
                                              n_replicas=CEIL_REPLICAS,
                                              n_production=TARGET_STEPS,
                                              cond=f'ceil_{name}', verbose=False)
        json.dump({'ceiling': ceiling}, open(ceil_f,'w'))
        print(f'  ceiling = {ceiling:.5f}')

    # --- fit ---
    print('  fitting ...')
    if spec['free']:
        # free C: gauge-fixed each iteration, lr_C = 3x lr_lambda, seed-sensitive
        best = None
        for seed in [3, 11, 42]:
            init = Model.free(N_BEADS, k=k, seed=seed); init.names=names; init.c=-0.30
            r = O.Optimizer(sim, target, lr_lambda=0.03, lr_C=0.10,
                            budget=O.Budget(n_steps=50_000, n_replicas=4),
                            out_dir=f'{OUT}/fits', tag=f'{name}_s{seed}',
                            verbose=True, quiet_sim=True
                            ).fit(init, n_iter=80, patience=15,
                                  history_path=f'{OUT}/fits/{name}_s{seed}_hist.json')
            rep = O.recovery_report(true_model, r.model)
            if best is None or rep['M_relative_error'] < best[1]['M_relative_error']:
                best = (r, rep, seed)
            if 'collapsed' not in r.stop_reason and rep['M_relative_error'] < 0.4:
                break
        res, rep, used_seed = best
        print(f'  best seed: {used_seed}')
    else:
        init = Model(C_true.copy(), spec['init_Lam'].copy(), c=-0.30, names=names)
        res = O.Optimizer(sim, target, lr_lambda=0.05,
                          budget=O.Budget(n_steps=50_000, n_replicas=4),
                          out_dir=f'{OUT}/fits', tag=name,
                          verbose=True, quiet_sim=True
                          ).fit(init, n_iter=60, patience=12,
                                history_path=f'{OUT}/fits/{name}_hist.json')
        rep = O.recovery_report(true_model, res.model)

    rec = dict(mode=name, k=int(k), n_lambda_params=int(k*(k+1)//2),
               ceiling=float(ceiling),
               M_relative_error=rep['M_relative_error'],
               M_scale_ratio=rep['M_scale_ratio'],
               spectrum_true=rep['spectrum_true'],
               spectrum_fitted=rep['spectrum_fitted'],
               Lam_true=np.asarray(Lam_true).tolist(),
               Lam_fitted=res.model.Lam.tolist(),
               final_loss=res.history[-1]['loss'],
               best_iter=res.best_iter, iterations=len(res.history),
               stop_reason=res.stop_reason,
               wall_minutes=(time.time()-t0)/60)
    if 'rotation_magnitude' in rep:
        rec['rotation_magnitude'] = rep['rotation_magnitude']
    results[name] = rec
    json.dump(results, open(RESULTS_F,'w'), indent=1, default=str)
    print(f'  -> M relerr {rec["M_relative_error"]:.4f}  '
          f'scale {rec["M_scale_ratio"]:.3f}  ({rec["wall_minutes"]:.0f} min)')
    return rec

for name in MODES_TO_RUN:
    run_mode(name)

## 3. Results

`M_relative_error` is primary. **Do not read `M_correlation`** — for k=1 it is
degenerate (the off-diagonal shape is proportional to CCᵀ for any λ, so it reads
±1.000 regardless of magnitude).

In [ ]:
if results:
    print(f"{'mode':11s} {'k':>2s} {'nΛ':>3s} {'relerr':>8s} {'scale':>7s} "
          f"{'ceiling':>8s} {'loss':>10s} {'min':>5s}")
    for n, r in results.items():
        print(f"{n:11s} {r['k']:2d} {r['n_lambda_params']:3d} "
              f"{r['M_relative_error']:8.4f} {r['M_scale_ratio']:7.3f} "
              f"{r['ceiling']:8.4f} {r['final_loss']:10.3e} {r['wall_minutes']:5.0f}")
else:
    print('no results yet')

### 3a. Λ recovery, entry by entry

For fixed C of full column rank there is **no gauge freedom**, so Λ should be
recoverable entry-by-entry. Discrepancies here are the data failing to determine
a parameter, not an ambiguity in the parameterization.

In [ ]:
for n, r in results.items():
    Lt = np.array(r['Lam_true']); Lf = np.array(r['Lam_fitted'])
    if Lt.size > 9: continue
    print(f'\n{n}:')
    print('  true  ', np.round(Lt,3).tolist())
    print('  fitted', np.round(Lf,3).tolist())
    print(f'  max |ΔΛ| = {np.abs(Lf-Lt).max():.4f}')
    if Lt.shape[0] >= 2:
        # per-mode: is the WEAKER mode recovered as well as the strong one?
        for i in range(Lt.shape[0]):
            rel = abs(Lf[i,i]-Lt[i,i]) / max(abs(Lt[i,i]), 1e-9)
            print(f'    mode {i}: true {Lt[i,i]:+.3f}  fitted {Lf[i,i]:+.3f}'
                  f'   rel err {rel:.1%}')

### 3b. The k=2 question

Compare `k2_strong` and `k2_weak`. If the strong second mode is recovered and the
weak one is not, you have **measured the detectability threshold for a second
mode** — the identifiability result for the multi-mode continuous model, and the
answer to whether the PC2 misalignment matters once Λ is fitted rather than
inherited.

In [ ]:
pair = [m for m in ('k2_strong','k2_weak') if m in results]
if len(pair) == 2:
    for m in pair:
        r = results[m]
        Lt, Lf = np.array(r['Lam_true']), np.array(r['Lam_fitted'])
        strength = abs(Lt[1,1]/Lt[0,0])
        rel1 = abs(Lf[1,1]-Lt[1,1])/max(abs(Lt[1,1]),1e-9)
        print(f'{m:10s} 2nd/1st mode strength = {strength:.2f}   '
              f'2nd mode recovered to {rel1:.0%}')
    print('\n-> a mode recovered well when strong and poorly when weak gives the')
    print('   detectability threshold. Both recovered = 2nd modes are usable.')
    print('   Neither = the data does not constrain a 2nd mode at this quality.')
else:
    print('run both k2_strong and k2_weak for this comparison')

### 3c. Precision vs parameter count

In [ ]:
if len(results) >= 2:
    xs = [r['n_lambda_params'] for r in results.values()]
    ys = [r['M_relative_error'] for r in results.values()]
    lb = list(results.keys())
    fig, ax = plt.subplots(1, 2, figsize=(11,3.6))
    ax[0].scatter(xs, ys); ax[0].set_xscale('log')
    for x,y,l in zip(xs,ys,lb): ax[0].annotate(l,(x,y),fontsize=8,
                                               xytext=(3,3),textcoords='offset points')
    ax[0].set_xlabel('Λ parameters'); ax[0].set_ylabel('M relative error')
    ax[0].set_title('precision vs parameter count')
    for n,r in results.items():
        st, sf = np.array(r['spectrum_true']), np.array(r['spectrum_fitted'])
        ax[1].plot(np.abs(st[:4]), 'o-', label=f'{n} true', alpha=.7)
        ax[1].plot(np.abs(sf[:4]), 's--', label=f'{n} fit', alpha=.7)
    ax[1].set_yscale('log'); ax[1].set_xlabel('eigenvalue index')
    ax[1].set_ylabel('|eigenvalue| of M'); ax[1].set_title('spectrum recovery')
    ax[1].legend(fontsize=6)
    plt.tight_layout(); plt.show()

## 4. What this establishes

- **which parameterizations are recoverable** at this data quality;
- **how precision degrades with parameter count** — the empirical cost of model
  complexity, which is the practical form of the low-rank argument;
- **whether a second continuous mode is determinable**, which decides if the
  multi-mode model is worth pursuing and whether PC2's misalignment with
  MiChroM's second axis matters once Λ is fitted rather than inherited;
- a **ceiling** from `freeC`: the best a rank-1 potential achieves with no
  biological prior, which is the comparison point for informed coordinates.

**Next:** Experiment A (identifiability vs data quality) — sweep replica count
and production length on the mode that recovers most cleanly, to turn the single
±0.2 measurement into a precision-versus-data curve.